# 02 — Worker Runner Template

Notebook ini adalah template worker untuk laptop pribadi atau PC worker lain.

Jalankan di **kernel berbeda** dari Optuna orchestrator. Worker tidak membuat trial; worker hanya:
- register hardware ke PostgreSQL lokal `cqcnn_orchestration`;
- claim task `WAITING`;
- ambil parameter trial dari Optuna PostgreSQL;
- training;
- kirim heartbeat;
- upload/register checkpoint;
- `study.tell()` ke Optuna PostgreSQL;
- update task menjadi `TOLD` atau `DONE`.

Default notebook memakai `real_train_one_task()` untuk training asli. Set `DRY_RUN=1` di `.env` untuk test alur saja.

### 0. Universal Setup & Environment Diagnostic

Jalankan cell di bawah ini **pertama kali** (terutama jika Anda berada di platform *cloud* seperti **Kaggle, Google Colab, atau PC baru**). 
Sistem akan otomatis:
1. Membaca `requirements.txt` dan mengunduh semua library yang kurang.
2. Menjalankan skrip `check_environment.py` untuk memastikan (Deep Check) bahwa PyTorch, GPU, Database, dan Rclone Anda berfungsi secara riil.

> **TIPS PENTING**: Jika *pip install* baru saja menginstal banyak hal untuk pertama kalinya, Anda **Wajib melakukan Restart Kernel** (Menu bar: Kernel -> Restart) agar Jupyter bisa mendeteksi *library* yang baru diunduh. Setelah *restart*, jalankan ulang sel-sel di bawahnya.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
req_path = PROJECT_ROOT / "requirements.txt"
check_path = PROJECT_ROOT / "check_environment.py"

print("1. Memastikan semua dependensi terinstall...")
!{sys.executable} -m pip install -q -r {req_path}

print("\n2. Menjalankan Uji Coba Kesiapan Environment...")
!{sys.executable} {check_path}

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PY_DIR = PROJECT_ROOT / "patched_program_files"
if str(PY_DIR) not in sys.path:
    sys.path.insert(0, str(PY_DIR))

from env_loader import load_dotenv_if_exists, require_env
from postgres_orchestration_db import PostgresOrchestrationDb, PostgresOrchestrationDbConfig
from worker_hardware_profile import detect_worker_hardware
from worker_loop import run_worker_loop
from worker_task_template import train_one_task
from worker_runtime_config import (
    WorkerRuntimeProfile,
    create_cpu_profile,
    create_gpu_profile,
    with_seed,
    build_runtime_plan,
    detect_worker_resources,
    print_runtime_summary,
)

load_dotenv_if_exists(PROJECT_ROOT / ".env")
print("PROJECT_ROOT:", PROJECT_ROOT)

### 1. Konfigurasi Worker

Edit parameter di bawah ini **setiap kali Anda berpindah PC/cloud** atau mengubah stage yang sedang dilatih.

In [ ]:
# === EDIT PARAMETER INI DI SETIAP PC ===
WORKER_UID = "laptop_pribadi_01"  # ganti misalnya: "pc_worker_02"
WORKER_NAME = "Laptop Pribadi"
WORKER_TYPE = "LOCAL_PC"  # LOCAL_PC, LAPTOP, LAB_PC, CLOUD_CPU, CLOUD_GPU

STAGE_NO = 1  # Stage Berurutan 1,2,3,4,5
MODEL_TYPE = "classical_fully_spatial"  # classical_fully_spatial | hybrid_qcqcnn

POLL_SECONDS = 30
STALE_AFTER = "15 minutes"
ALLOW_HIJACK = True

ORCHESTRATION_DB_DSN = require_env("ORCHESTRATION_DB_DSN")
OPTUNA_STORAGE_URL = require_env("OPTUNA_STORAGE_URL")

### 2. Seed & Hardware Mode (Terpisah)

Seed dan mode hardware **dipisahkan** agar bisa di-set secara independen setiap kali notebook dijalankan.

| `HARDWARE_MODE` | Kapan Pakai | Contoh Platform |
|---|---|---|
| `"auto"` | Biarkan sistem mendeteksi otomatis | Default, aman di mana saja |
| `"cpu"` | Paksa CPU (misal: Colab mode CPU, laptop tanpa GPU) | Laptop, Colab CPU |
| `"gpu"` | Paksa GPU (misal: Colab GPU, Kaggle P100, PC NVIDIA) | Colab GPU, T4, A100 |

**Micro batch** ditentukan otomatis berdasarkan:
- **GPU**: tier VRAM (4/8/12/16/24 GB)
- **CPU**: tier RAM usable + jumlah physical cores

**Gradient accumulation steps** dihitung dari `global_batch / micro_batch`.

In [ ]:
# === SEED TERPISAH (bisa diubah tanpa mengubah hardware) ===
SEED = 42

# === PILIH MODE HARDWARE ===
HARDWARE_MODE = "auto"  # "auto" | "cpu" | "gpu"

# Build runtime profile berdasarkan pilihan di atas
if HARDWARE_MODE == "cpu":
    runtime_profile = create_cpu_profile(worker_id=WORKER_UID, seed=SEED)
elif HARDWARE_MODE == "gpu":
    runtime_profile = create_gpu_profile(worker_id=WORKER_UID, seed=SEED)
else:
    runtime_profile = with_seed(
        WorkerRuntimeProfile(worker_id=WORKER_UID),
        seed=SEED,
    )

print(f"Hardware Mode  : {HARDWARE_MODE}")
print(f"Seed           : {SEED}")
print(f"Profile        : torch_mode={runtime_profile.torch_mode}, quantum_mode={runtime_profile.quantum_mode}")
print(f"Global Batch   : {runtime_profile.global_batch_size}")

### 3. Register Worker & Preview Runtime Plan

Cell ini akan:
1. Mendeteksi hardware real-time (GPU VRAM, RAM, CPU cores)
2. Mendaftarkan worker ke database orchestration
3. Menampilkan **Runtime Plan** per model:
   - `micro_batch_size` — ukuran batch fisik yang masuk ke GPU/CPU per step
   - `gradient_accumulation_steps` — berapa micro-batch diakumulasi sebelum optimizer.step()
   - `global_batch_size` — ukuran batch virtual = micro × accum

In [ ]:
db = PostgresOrchestrationDb(PostgresOrchestrationDbConfig(dsn=ORCHESTRATION_DB_DSN))
print("orchestration connection:", db.test_connection())
print("stage_information rows:", len(db.query_stage_information()))

hw_profile = detect_worker_hardware(
    worker_uid=WORKER_UID,
    worker_name=WORKER_NAME,
    worker_type=WORKER_TYPE,
)
worker_id = db.register_worker(**hw_profile.to_orchestration_kwargs())
print("Registered worker_id:", worker_id)
print()
print("=" * 55)
print("HARDWARE PROFILE")
print("=" * 55)
print(f"  CPU          : {hw_profile.cpu_name} ({hw_profile.cpu_count} cores)")
print(f"  RAM          : {hw_profile.ram_gb} GB")
print(f"  GPU          : {hw_profile.gpu_name or 'N/A'}")
print(f"  GPU VRAM     : {hw_profile.gpu_vram_gb or 'N/A'} GB")
print(f"  Platform     : {hw_profile.platform_name}")
print()

# Preview runtime plan untuk model yang sedang dilatih
resources = detect_worker_resources(runtime_profile)
plan = build_runtime_plan(MODEL_TYPE, runtime_profile, resources)
print("=" * 55)
print(f"RUNTIME PLAN ({MODEL_TYPE})")
print("=" * 55)
print(f"  Torch Device         : {plan.torch_device}")
print(f"  Micro Batch Size     : {plan.micro_batch_size}")
print(f"  Gradient Accum Steps : {plan.gradient_accumulation_steps}")
print(f"  Global Batch Size    : {plan.global_batch_size}")
print(f"  DataLoader Workers   : {plan.dataloader_workers}")
print(f"  Quantum Device       : {plan.quantum_device}")
print(f"  Quantum Diff Method  : {plan.quantum_diff_method}")
print(f"  Notes                : {plan.notes}")

### 4. Start Worker Loop

Jalankan cell ini dan **biarkan berjalan**. Worker akan:
- Mengambil task `WAITING` dari database orchestration
- Training dengan gradient accumulation (global batch sesuai runtime plan)
- Upload checkpoint ke Google Drive via rclone
- Berhenti otomatis saat `STAGE_COMPLETE`

In [ ]:
# Run cell ini dan biarkan berjalan.
# Worker akan idle jika belum ada task dan berhenti saat STAGE_COMPLETE.
run_worker_loop(
    db=db,
    worker_uid=WORKER_UID,
    stage_no=STAGE_NO,
    model_type=MODEL_TYPE,
    train_one_task=train_one_task,
    optuna_storage_url=OPTUNA_STORAGE_URL,
    poll_seconds=POLL_SECONDS,
    stale_after=STALE_AFTER,
    allow_hijack=ALLOW_HIJACK,
    stop_when_stage_complete=True,
)